In [1]:
import cytnx
import numpy as np

In [2]:
path = '../Networks'

def ut_print(ut, print_numpy=True):
    ut.print_diagram()
    if print_numpy:
        print(ut.get_block().numpy())

def get_M(T):
    W = np.array([[np.exp(1/T), np.exp(-1/T)],
                  [np.exp(-1/T), np.exp(1/T)]])
    W = cytnx.from_numpy(W)
    S, U, Vd = cytnx.linalg.Svd(W)
    M = U @ cytnx.linalg.Diag(S.Pow(0.5))
    M = cytnx.UniTensor(M, rowrank=1)
    M.set_name('M')
    Md = cytnx.linalg.Diag(S.Pow(0.5)) @ Vd
    Md = cytnx.UniTensor(Md, rowrank=1)
    Md.set_name('Md')
    return M, Md

def get_delta(T, h):
    delta = cytnx.zeros([2, 2, 2, 2])
    delta[0, 0, 0, 0] = np.exp(-h/T)
    delta[1, 1, 1, 1] = np.exp(h/T)
    delta = cytnx.UniTensor(delta, rowrank=2)
    delta.set_name('delta')
    return delta

def get_T_baret(T, h=0):
    ut_M, ut_Md = get_M(T)
    ut_delta = get_delta(T, h)
    Ising_net = cytnx.Network(f'{path}/Ising_square.net')
    Ising_net.PutUniTensors(['delta', 'M0.d', 'M1.d', 'M2', 'M3'],
                            [ut_delta, ut_Md, ut_Md, ut_M, ut_M])
    T_bare = Ising_net.Launch()
    T_bare.set_name('T_bare')
    return T_bare

In [3]:
print(get_T_baret(4, h = 0.1).get_block().numpy())

[[[[ 2.12829088 -0.0263264 ]
   [-0.0263264   0.52125816]]

  [[-0.0263264   0.52125816]
   [ 0.52125816 -0.00644783]]]


 [[[-0.0263264   0.52125816]
   [ 0.52125816 -0.00644783]]

  [[ 0.52125816 -0.00644783]
   [-0.00644783  0.12766585]]]]


In [4]:
help(cytnx.linalg.Svd)

Help on built-in function Svd in module cytnx.cytnx.linalg:

Svd(...) method of builtins.PyCapsule instance
    Svd(*args, **kwargs)
    Overloaded function.
    
    1. Svd(Tin: cytnx.cytnx.Tensor, is_UvT: bool = True) -> list[cytnx.cytnx.Tensor]
    
    2. Svd(Tin: cytnx.cytnx.UniTensor, is_UvT: bool = True) -> list[cytnx.cytnx.UniTensor]



In [25]:
def merge_y(Tup, Tdn, combine=False, trace=False):
    if trace:
        TupTdn_net = cytnx.Network(f'{path}/merge_y_trace.net')
    else:
        TupTdn_net = cytnx.Network(f'{path}/merge_y.net')
    TupTdn_net.PutUniTensors(['Tup', 'Tdn'], [Tup, Tdn])
    TupTdn = TupTdn_net.Launch()
    if combine:
        TupTdn.combineBonds(['1', '2'])
        TupTdn.combineBonds(['3', '4'])
    return TupTdn

def merge_y_truncate(Tup, Tdn, dcut):
    if ((Tup.shape()[1] * Tdn.shape()[1]) < dcut):
        return merge_y(Tup, Tdn, True)
    TupTdn_pure_net = cytnx.Network(f'{path}/merge_y_pure.net')
    TupTdn_pure_net.PutUniTensors(['Tup', 'Tdn', 'Tupd', 'Tdnd'],
                                  [Tup, Tdn, Tup, Tdn])
    TupTdn_pure = TupTdn_pure_net.Launch()
    _, U, __ = cytnx.linalg.Svd_truncate(TupTdn_pure, dcut)
    TupTdn_net = cytnx.Network(f'{path}/merge_y_truncate.net')
    TupTdn_net.PutUniTensors(['Tup', 'Tdn', 'UL', 'UR'],
                             [Tup, Tdn, U, U])
    TupTdn = TupTdn_net.Launch()
    return TupTdn

In [26]:
temp = 2
dcut = 8

TL = get_T_baret(temp)
print(TL)
TL = merge_y_truncate(TL,TL,dcut)
print(TL)
TL = merge_y_truncate(TL,TL,dcut)
print(TL)
TL = merge_y_truncate(TL,TL,dcut)
print(TL)
TL = merge_y_truncate(TL,TL,dcut)
print(TL)

-------- start of print ---------
Tensor name: T_bare
is_diag    : False
contiguous : True

Total elem: 16
type  : Double (Float64)
cytnx device: CPU
Shape : (2,2,2,2)
[[[[2.54308e+00 2.22045e-16 ]
   [3.33067e-16 1.17520e+00 ]]
  [[3.33067e-16 1.17520e+00 ]
   [1.17520e+00 1.66533e-16 ]]]
 [[[3.33067e-16 1.17520e+00 ]
   [1.17520e+00 1.66533e-16 ]]
  [[1.17520e+00 0.00000e+00 ]
   [0.00000e+00 5.43081e-01 ]]]]




-------- start of print ---------
Tensor name: 
is_diag    : False
contiguous : True

Total elem: 64
type  : Double (Float64)
cytnx device: CPU
Shape : (2,4,4,2)
[[[[6.46726e+00 8.25624e-16 ]
   [1.10796e-15 2.98863e+00 ]
   [1.23844e-15 1.38110e+00 ]
   [1.38110e+00 5.87131e-16 ]]
  [[1.10796e-15 2.98863e+00 ]
   [2.98863e+00 5.44096e-16 ]
   [1.38110e+00 3.91421e-16 ]
   [3.91421e-16 6.38229e-01 ]]
  [[1.23844e-15 1.38110e+00 ]
   [1.38110e+00 5.87131e-16 ]
   [2.98863e+00 4.56657e-16 ]
   [5.87131e-16 1.38110e+00 ]]
  [[1.38110e+00 3.91421e-16 ]
   [3.91421e-16 6.38229e-0

   [4.95820e+01 1.20626e-13 ]
   [-2.47910e-12 -1.87823e+02 ]
   [-6.45700e+02 -5.90113e-13 ]
   [7.38013e-13 1.70219e-13 ]
   [-3.40236e+01 -3.26711e-13 ]]
  [[8.20180e-12 3.63833e-14 ]
   [4.29358e+02 -1.03920e-12 ]
   [-5.91029e-01 -4.51386e-13 ]
   [8.66495e-13 5.67816e-14 ]
   [-6.45700e+02 -5.90113e-13 ]
   [2.26324e-12 -4.85998e+02 ]
   [2.90177e+00 2.49016e-13 ]
   [-4.21441e-13 -4.20563e-14 ]]
  [[2.82805e+02 -1.51628e-12 ]
   [-7.16608e-12 8.17124e-14 ]
   [-5.20145e-13 -9.25820e+01 ]
   [-3.67733e+01 1.09109e-12 ]
   [7.07886e-13 1.61871e-13 ]
   [2.90177e+00 2.32246e-13 ]
   [5.61860e-12 -2.74694e+02 ]
   [-4.88096e+02 -4.01376e-13 ]]
  [[5.43185e-12 2.51283e+02 ]
   [2.03626e+02 -4.92660e-12 ]
   [2.16567e+01 -9.02483e-13 ]
   [-4.14873e-13 -1.01961e+02 ]
   [-3.40236e+01 -3.52325e-13 ]
   [-4.58231e-13 -1.77636e-14 ]
   [-4.88096e+02 -4.01376e-13 ]
   [-5.99065e-12 -2.41005e+02 ]]]]




-------- start of print ---------
Tensor name: 
is_diag    : False
contiguous : False


In [31]:
def merge_x(TL, TR, combine=False, trace=False):
    if trace:
        n = TL.shape()[0]
        I = np.eye(n)
        I = cytnx.from_numpy(I)
        I = cytnx.UniTensor(I, rowrank=1)
        TLTR_net = cytnx.Network(f'{path}/merge_x_trace.net')
        TLTR_net.PutUniTensors(['TL', 'TR', 'IL', 'IR'], [TL, TR, I, I])
        TLTR = TLTR_net.Launch()
    else:
        TLTR_net = cytnx.Network(f'{path}/merge_x.net')
        TLTR_net.PutUniTensors(['TL', 'TR'], [TL, TR])
        TLTR = TLTR_net.Launch()
    if combine and not trace:
        TLTR.combineBonds(['0', '1'])
        TLTR.combineBonds(['4', '5'])
    return TLTR

def merge_x_truncate(TL, TR, dcut):
    if ((TL.shape()[0] * TR.shape()[0]) < dcut):
        return merge_x(TL, TR, True)
    TLTR_pure_net = cytnx.Network(f'{path}/merge_x_pure.net')
    TLTR_pure_net.PutUniTensors(['TL', 'TR', 'TLd', 'TRd'],
                                [TL, TR, TL, TR])
    TLTR_pure = TLTR_pure_net.Launch()
    try:
        _, U, __ = cytnx.linalg.Svd_truncate(TLTR_pure, dcut)
    except:
        TLTR_pure.print_diagram()
        _, U, __ = cytnx.linalg.Svd(TLTR_pure, dcut)
    TLTR_net = cytnx.Network(f'{path}/merge_x_truncate.net')
    TLTR_net.PutUniTensors(['TL', 'TR', 'Uup', 'Udn'], [TL, TR, U, U])
    TLTR = TLTR_net.Launch()
    return TLTR

TL = get_T_baret(temp)
print(TL)
TL = merge_x(TL,TL,True,False)
print(TL)

-------- start of print ---------
Tensor name: T_bare
is_diag    : False
contiguous : True

Total elem: 16
type  : Double (Float64)
cytnx device: CPU
Shape : (2,2,2,2)
[[[[2.54308e+00 2.22045e-16 ]
   [3.33067e-16 1.17520e+00 ]]
  [[3.33067e-16 1.17520e+00 ]
   [1.17520e+00 1.66533e-16 ]]]
 [[[3.33067e-16 1.17520e+00 ]
   [1.17520e+00 1.66533e-16 ]]
  [[1.17520e+00 0.00000e+00 ]
   [0.00000e+00 5.43081e-01 ]]]]




-------- start of print ---------
Tensor name: 
is_diag    : False
contiguous : True

Total elem: 64
type  : Double (Float64)
cytnx device: CPU
Shape : (4,2,2,4)
[[[[6.46726e+00 9.56098e-16 9.56098e-16 1.38110e+00 ]
   [1.23844e-15 1.38110e+00 2.98863e+00 4.56657e-16 ]]
  [[1.23844e-15 2.98863e+00 1.38110e+00 4.56657e-16 ]
   [1.38110e+00 5.87131e-16 5.87131e-16 1.38110e+00 ]]]
 [[[1.23844e-15 2.98863e+00 1.38110e+00 4.56657e-16 ]
   [1.38110e+00 5.87131e-16 5.87131e-16 1.38110e+00 ]]
  [[2.98863e+00 1.80882e-16 2.60947e-16 6.38229e-01 ]
   [3.91421e-16 6.38229e-01 1.38110e+

In [32]:
temp = 2
dcut = 2

TL = get_T_baret(temp)
print(TL)
TL = merge_x_truncate(TL,TL,dcut)
print(TL)
TL = merge_x_truncate(TL,TL,dcut)
print(TL)
TL = merge_x_truncate(TL,TL,dcut)
print(TL)
TL = merge_x_truncate(TL,TL,dcut)
print(TL)

-------- start of print ---------
Tensor name: T_bare
is_diag    : False
contiguous : True

Total elem: 16
type  : Double (Float64)
cytnx device: CPU
Shape : (2,2,2,2)
[[[[2.54308e+00 2.22045e-16 ]
   [3.33067e-16 1.17520e+00 ]]
  [[3.33067e-16 1.17520e+00 ]
   [1.17520e+00 1.66533e-16 ]]]
 [[[3.33067e-16 1.17520e+00 ]
   [1.17520e+00 1.66533e-16 ]]
  [[1.17520e+00 0.00000e+00 ]
   [0.00000e+00 5.43081e-01 ]]]]




-------- start of print ---------
Tensor name: 
is_diag    : False
contiguous : False

Total elem: 16
type  : Double (Float64)
cytnx device: CPU
Shape : (2,2,2,2)
[[[[6.79798e+00 -1.65728e-15 ]
   [9.26032e-15 3.37721e+00 ]]
  [[9.28949e-15 3.37721e+00 ]
   [2.07880e+00 5.19906e-16 ]]]
 [[[-1.33264e-15 4.36973e+00 ]
   [3.37721e+00 -7.37811e-15 ]]
  [[3.37721e+00 -7.68772e-15 ]
   [2.94874e-16 2.01933e+00 ]]]]




-------- start of print ---------
Tensor name: 
is_diag    : False
contiguous : False

Total elem: 16
type  : Double (Float64)
cytnx device: CPU
Shape : (2,2,2,2)


In [34]:
MaxL = 4
Temp = np.linspace(2.26,4,100)
W = np.ones((MaxL+1,len(Temp),2))
E = np.ones((MaxL+1,len(Temp),2))
Tc = 2/np.log(1+np.sqrt(2))

In [35]:
%%time
dcut = 3

for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    print(TL)
    for j in range(2,MaxL+1):
        TLx = merge_x_truncate(TL,TL,dcut)
        print(TLx)
        TL_Trace = merge_y(TLx,TLx,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        print(TL)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
        print(TL)
    break
E = -np.log(W)
Corr_len_2 = 1/(E[:,:,-2]-E[:,:,-1])

-------- start of print ---------
Tensor name: T_bare
is_diag    : False
contiguous : True

Total elem: 16
type  : Double (Float64)
cytnx device: CPU
Shape : (2,2,2,2)
[[[[2.41780e+00 7.77156e-16 ]
   [6.66134e-16 1.00507e+00 ]]
  [[2.22045e-16 1.00507e+00 ]
   [1.00507e+00 1.11022e-16 ]]]
 [[[2.22045e-16 1.00507e+00 ]
   [1.00507e+00 1.11022e-16 ]]
  [[1.00507e+00 -1.11022e-16 ]
   [-1.11022e-16 4.17805e-01 ]]]]




-------- start of print ---------
Tensor name: 
is_diag    : False
contiguous : False

Total elem: 36
type  : Double (Float64)
cytnx device: CPU
Shape : (3,2,2,3)
[[[[6.03633e+00 1.42412e-16 -4.09153e-17 ]
   [8.94878e-15 2.60554e+00 8.72328e-01 ]]
  [[7.25537e-15 2.60554e+00 -8.72328e-01 ]
   [1.43744e+00 8.17440e-16 -6.49610e-16 ]]]
 [[[-1.50671e-15 3.44024e+00 5.77316e-15 ]
   [2.60554e+00 -5.12224e-15 -1.73371e-15 ]]
  [[2.60554e+00 -5.81820e-15 1.08654e-15 ]
   [5.43225e-16 1.43009e+00 5.44009e-15 ]]]
 [[[-4.09153e-17 5.77316e-15 1.41990e+00 ]
   [8.72328e-01 -1.39895

   [2.40583e-13 1.63568e+01 2.44249e-15 ]]
  [[-2.33181e+01 2.00760e-13 2.72822e-14 ]
   [-3.94211e-13 -1.63568e+01 -4.10783e-15 ]
   [-1.79363e-14 6.65565e-15 6.38102e+00 ]]]]




-------- start of print ---------
Tensor name: 
is_diag    : False
contiguous : False

Total elem: 81
type  : Double (Float64)
cytnx device: CPU
Shape : (3,3,3,3)
[[[[2.70903e+04 1.48296e-09 6.54289e-12 ]
   [4.00637e-10 1.59150e+04 4.15812e+03 ]
   [1.28197e-11 4.89331e+03 4.88403e+03 ]]
  [[3.55609e-10 1.59150e+04 -4.15812e+03 ]
   [1.31088e+04 -1.35713e-09 -3.17638e-11 ]
   [-2.54491e-13 -2.28073e-10 -1.37950e-10 ]]
  [[1.28197e-11 4.89331e+03 -4.88403e+03 ]
   [1.88704e-12 -2.16211e-10 9.97881e-11 ]
   [1.77677e+03 -8.67974e-11 -6.42458e-12 ]]]
 [[[1.44041e-09 1.19018e+04 2.77899e-12 ]
   [1.59150e+04 -5.30290e-10 -8.17670e-11 ]
   [-4.89331e+03 -4.18828e-11 7.00132e-11 ]]
  [[1.59150e+04 -5.33049e-10 6.53574e-11 ]
   [-1.36287e-09 1.27727e+04 -9.66338e-13 ]
   [2.07735e-10 1.52598e-12 3.71618e+03 ]]
  [

<timed exec>:19: RuntimeWarning: divide by zero encountered in divide
